# EDA – Conectividad fija en Argentina (2014–2024)
**Integrante 2: análisis exploratorio, estadísticas, gráficos e hipótesis**

Este notebook cubre estructura, calidad inicial, estadística descriptiva, distribuciones, variables categóricas y numéricas, cruces, correlaciones, evolución temporal, patrones, outliers y conclusiones exploratorias.

> **Importante:** el análisis parte de las tablas `fact_*` y dimensiones del modelo SQLite. El notebook inspecciona el esquema y propone columnas automáticamente, pero hay una sección de configuración para corregir los nombres si el reconocimiento no coincide con tu base. No modifica la base ni elimina outliers.


## 1. Configuración

In [ ]:
from pathlib import Path
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
warnings.filterwarnings("ignore", category=FutureWarning)

# Ajustá esta ruta si tu base está en otra ubicación.
DB_PATH = Path("../data/conectividad.db")

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró la base en: {DB_PATH.resolve()}\n"
        "Modificá DB_PATH para apuntar a tu archivo .db o .sqlite."
    )

conn = sqlite3.connect(DB_PATH)
print("Base conectada:", DB_PATH.resolve())


## 2. Revisión de estructura del dataset y esquema disponible

In [ ]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    conn
)
views = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name",
    conn
)
display(tables)
display(views)

def table_columns(table_name):
    return pd.read_sql_query(f'PRAGMA table_info("{table_name}")', conn)

schema_rows = []
for table_name in tables["name"]:
    info = table_columns(table_name)
    for _, row in info.iterrows():
        schema_rows.append({
            "tabla": table_name,
            "columna": row["name"],
            "tipo": row["type"],
            "not_null": bool(row["notnull"]),
            "primary_key": bool(row["pk"])
        })
schema = pd.DataFrame(schema_rows)
display(schema)


### 2.1 Volumen de registros por tabla

In [ ]:
row_counts = []
for table_name in tables["name"]:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM "{table_name}"', conn).iloc[0, 0]
    row_counts.append({"tabla": table_name, "filas": int(n)})
row_counts = pd.DataFrame(row_counts).sort_values("filas", ascending=False)
display(row_counts)
row_counts.plot(kind="barh", x="tabla", y="filas", legend=False, figsize=(10, max(4, len(row_counts)*0.3)))
plt.title("Cantidad de registros por tabla")
plt.xlabel("Filas")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 3. Carga de tablas de hechos

Se cargan las tablas de hechos principales. Si una tabla no existe, se informa y el resto del notebook puede continuar cuando no dependa de ella.

In [ ]:
FACT_TABLES = [
    "fact_accesos_provincia",
    "fact_accesos_tecnologia",
    "fact_velocidad_provincia",
    "fact_penetracion_provincia",
    "fact_accesos_categoria_provincia",
    "fact_accesos_tecnologia_localidad",
    "fact_accesos_categoria_localidad",
]

data = {}
for t in FACT_TABLES:
    if t in set(tables["name"]):
        data[t] = pd.read_sql_query(f'SELECT * FROM "{t}"', conn)
        print(f"{t}: {len(data[t]):,} filas | {len(data[t].columns)} columnas")
    else:
        print(f"AVISO: no existe {t}")

for t, df in data.items():
    print("\n", t, "->", list(df.columns))


## 4. Perfil de calidad inicial: nulos, duplicados y tipos

Los duplicados se revisan sobre filas completas. La clave lógica puede ser diferente según cada tabla, por lo que esta revisión no reemplaza las vistas `quality_*` del SQL.

In [ ]:
quality_profile = []
for t, df in data.items():
    quality_profile.append({
        "tabla": t,
        "filas": len(df),
        "columnas": len(df.columns),
        "celdas_nulas": int(df.isna().sum().sum()),
        "porcentaje_celdas_nulas": round(df.isna().sum().sum() / max(df.size, 1) * 100, 2),
        "filas_duplicadas_completas": int(df.duplicated().sum())
    })
display(pd.DataFrame(quality_profile))

for t, df in data.items():
    print(f"\n{t} – porcentaje de nulos por columna")
    display((df.isna().mean().mul(100).round(2).rename("porcentaje_nulos")
             .to_frame().query("porcentaje_nulos > 0").sort_values("porcentaje_nulos", ascending=False)))


## 5. Configuración de variables de análisis

El siguiente bloque intenta reconocer columnas habituales. **Revisá los resultados impresos**. Si algún campo queda como `None` o se asigna incorrectamente, completá manualmente el diccionario `MAP` usando los nombres que aparecieron en la sección 3.

No se asume que las categorías de velocidad tengan rangos numéricos: se conservan como etiquetas originales.

In [ ]:
def choose_col(df, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    # segundo intento: coincidencia parcial
    for candidate in candidates:
        for c in df.columns:
            if candidate.lower() in str(c).lower():
                return c
    return None

MAP = {
    "fact_accesos_provincia": {
        "periodo": ["periodo_id", "periodo", "fecha", "trimestre"],
        "provincia": ["provincia_id", "provincia"],
        "accesos": ["accesos_total", "total_accesos", "accesos"]
    },
    "fact_accesos_tecnologia": {
        "periodo": ["periodo_id", "periodo", "fecha", "trimestre"],
        "provincia": ["provincia_id", "provincia"],
        "tecnologia": ["tecnologia_id", "tecnologia"],
        "accesos": ["accesos", "cantidad_accesos", "total"]
    },
    "fact_velocidad_provincia": {
        "periodo": ["periodo_id", "periodo", "fecha", "trimestre"],
        "provincia": ["provincia_id", "provincia"],
        "velocidad": ["mbps_media", "velocidad_media", "velocidad"]
    },
    "fact_penetracion_provincia": {
        "periodo": ["periodo_id", "periodo", "fecha", "trimestre"],
        "provincia": ["provincia_id", "provincia"],
        "hogares": ["accesos_cada_100_hogares", "accesos_por_100_hogares"],
        "habitantes": ["accesos_cada_100_habitantes", "accesos_por_100_habitantes"]
    },
    "fact_accesos_categoria_provincia": {
        "periodo": ["periodo_id", "periodo", "fecha", "trimestre"],
        "provincia": ["provincia_id", "provincia"],
        "categoria": ["categoria_velocidad_id", "categoria", "velocidad_categoria"],
        "accesos": ["accesos", "cantidad_accesos", "total"]
    },
    "fact_accesos_tecnologia_localidad": {
        "provincia": ["provincia_id", "provincia"],
        "localidad": ["localidad_id", "localidad"],
        "tecnologia": ["tecnologia_id", "tecnologia"],
        "accesos": ["accesos", "cantidad_accesos", "total"]
    },
    "fact_accesos_categoria_localidad": {
        "provincia": ["provincia_id", "provincia"],
        "localidad": ["localidad_id", "localidad"],
        "categoria": ["categoria_velocidad_id", "categoria", "velocidad_categoria"],
        "accesos": ["accesos", "cantidad_accesos", "total"]
    }
}

for table, spec in MAP.items():
    if table not in data:
        continue
    print("\n", table)
    for role, candidates in spec.items():
        detected = choose_col(data[table], candidates)
        print(f"  {role:12s} -> {detected}")


### 5.1 Enriquecer claves con dimensiones (si corresponde)

Si los hechos guardan IDs, se incorporan nombres de provincia, período, tecnología, categoría y localidad mediante las dimensiones. Si ya guardan nombres, se conservan. La función muestra qué dimensiones encontró.

In [ ]:
def dim_lookup(fact_df, fact_key_candidates, dim_table, dim_key_candidates, dim_value_candidates, output_name):
    if dim_table not in set(tables["name"]):
        return fact_df
    fk = choose_col(fact_df, fact_key_candidates)
    dim = pd.read_sql_query(f'SELECT * FROM "{dim_table}"', conn)
    dk = choose_col(dim, dim_key_candidates)
    dv = choose_col(dim, dim_value_candidates)
    if not all([fk, dk, dv]) or fk not in fact_df.columns:
        return fact_df
    if fk == output_name or output_name in fact_df.columns:
        return fact_df
    lookup = dim[[dk, dv]].drop_duplicates(dk).rename(columns={dk: fk, dv: output_name})
    return fact_df.merge(lookup, on=fk, how="left")

ENRICH = {
    "fact_accesos_provincia": [("provincia_id","dim_provincia",["provincia_id"],["provincia_id"],["provincia"])],
    "fact_accesos_tecnologia": [
        ("provincia_id","dim_provincia",["provincia_id"],["provincia_id"],["provincia"]),
        ("tecnologia_id","dim_tecnologia",["tecnologia_id"],["tecnologia_id"],["tecnologia"]),
        ("periodo_id","dim_periodo",["periodo_id"],["periodo_id"],["periodo","etiqueta","periodo_nombre"])
    ],
    "fact_velocidad_provincia": [
        ("provincia_id","dim_provincia",["provincia_id"],["provincia_id"],["provincia"]),
        ("periodo_id","dim_periodo",["periodo_id"],["periodo_id"],["periodo","etiqueta","periodo_nombre"])
    ],
    "fact_penetracion_provincia": [
        ("provincia_id","dim_provincia",["provincia_id"],["provincia_id"],["provincia"]),
        ("periodo_id","dim_periodo",["periodo_id"],["periodo_id"],["periodo","etiqueta","periodo_nombre"])
    ],
    "fact_accesos_categoria_provincia": [
        ("provincia_id","dim_provincia",["provincia_id"],["provincia_id"],["provincia"]),
        ("categoria_velocidad_id","dim_categoria_velocidad",["categoria_velocidad_id"],["categoria_velocidad_id"],["categoria","etiqueta","nombre"]),
        ("periodo_id","dim_periodo",["periodo_id"],["periodo_id"],["periodo","etiqueta","periodo_nombre"])
    ]
}
for t, joins in ENRICH.items():
    if t in data:
        for fk, dim_t, fact_keys, dim_keys, dim_values in joins:
            data[t] = dim_lookup(data[t], fact_keys, dim_t, dim_keys, dim_values, dim_t.replace("dim_",""))
        print(t, "columnas enriquecidas:", list(data[t].columns))


## 6. Estadísticas descriptivas de variables numéricas

Se resumen todas las columnas numéricas de las tablas de hechos: cantidad, media, mediana, desvío estándar, mínimo, cuartiles, máximo, asimetría y porcentaje de nulos.

In [ ]:
numeric_summaries = []
for t, df in data.items():
    num = df.select_dtypes(include="number")
    for c in num.columns:
        s = num[c].dropna()
        numeric_summaries.append({
            "tabla": t, "variable": c, "n": int(s.count()),
            "media": s.mean(), "mediana": s.median(), "desvio_std": s.std(),
            "min": s.min(), "q1": s.quantile(.25), "q3": s.quantile(.75),
            "max": s.max(), "asimetria": s.skew(),
            "nulos_pct": df[c].isna().mean()*100
        })
numeric_stats = pd.DataFrame(numeric_summaries)
display(numeric_stats)


## 7. Distribución de variables numéricas

Se generan histogramas y boxplots para las columnas numéricas. Para no producir gráficos ilegibles, se limita a las variables numéricas con datos y hasta 12 variables por tabla. Los IDs también pueden ser numéricos: no deben interpretarse como magnitudes.

In [ ]:
ID_HINTS = ("_id", "id_", "codigo", "cod")
for t, df in data.items():
    numeric_cols = [c for c in df.select_dtypes(include="number").columns
                    if not any(h in c.lower() for h in ID_HINTS)]
    for c in numeric_cols[:12]:
        s = df[c].dropna()
        if s.empty:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
        axes[0].hist(s, bins=30)
        axes[0].set_title(f"Distribución – {t}.{c}")
        axes[0].set_xlabel(c)
        axes[0].set_ylabel("Frecuencia")
        axes[1].boxplot(s, vert=False)
        axes[1].set_title(f"Boxplot – {c}")
        axes[1].set_xlabel(c)
        plt.tight_layout()
        plt.show()


## 8. Análisis de variables categóricas

Se identifican columnas de texto y se calculan frecuencias, porcentajes y cardinalidad. Para columnas con muchas categorías se muestran las 20 más frecuentes.

In [ ]:
categorical_summary = []
for t, df in data.items():
    for c in df.select_dtypes(include=["object", "category"]).columns:
        s = df[c].fillna("(nulo)").astype(str)
        categorical_summary.append({
            "tabla": t, "variable": c,
            "categorias_distintas": s.nunique(),
            "categoria_mas_frecuente": s.value_counts().index[0] if len(s) else None,
            "frecuencia_maxima": int(s.value_counts().iloc[0]) if len(s) else 0
        })
        print(f"\n{t}.{c} – {s.nunique()} categorías")
        display((s.value_counts().head(20).rename("frecuencia")
                 .to_frame().assign(porcentaje=lambda x: x["frecuencia"]/len(s)*100)))
display(pd.DataFrame(categorical_summary))


## 9. Evolución temporal de accesos totales

La consulta usa `fact_accesos_provincia` y suma los accesos de las provincias por período. La columna de período se intenta resolver con la dimensión. Si el período no quedó enriquecido, corregí `PERIOD_COL` manualmente.

In [ ]:
def get_col(table, role):
    if table not in data or table not in MAP or role not in MAP[table]:
        return None
    return choose_col(data[table], MAP[table][role])

def resolve_period_column(df):
    return choose_col(df, ["periodo", "periodo_nombre", "etiqueta", "trimestre", "fecha", "periodo_id"])

if "fact_accesos_provincia" in data:
    df = data["fact_accesos_provincia"].copy()
    access_col = choose_col(df, ["accesos_total", "total_accesos", "accesos"])
    period_col = resolve_period_column(df)
    if access_col and period_col:
        temporal = df.groupby(period_col, dropna=False)[access_col].sum().reset_index()
        temporal = temporal.rename(columns={period_col:"periodo", access_col:"accesos_total"})
        display(temporal.head())
        plt.figure(figsize=(13,5))
        plt.plot(temporal["periodo"].astype(str), temporal["accesos_total"], marker="o")
        plt.title("Evolución trimestral de accesos fijos")
        plt.xlabel("Período")
        plt.ylabel("Accesos")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()
    else:
        print("No se identificó período o accesos. Revisá las columnas de fact_accesos_provincia.")


## 10. Transformación tecnológica y participación de fibra

Se calcula la composición por tecnología y la participación de fibra óptica cuando los campos están disponibles. El total del hecho provincial no se mezcla con la composición tecnológica para evitar doble conteo.

In [ ]:
if "fact_accesos_tecnologia" in data:
    df = data["fact_accesos_tecnologia"].copy()
    tech_col = choose_col(df, ["tecnologia", "nombre_tecnologia"])
    acc_col = choose_col(df, ["accesos", "cantidad_accesos", "total"])
    per_col = resolve_period_column(df)
    if tech_col and acc_col and per_col:
        tech_time = df.groupby([per_col, tech_col], dropna=False)[acc_col].sum().reset_index()
        pivot = tech_time.pivot(index=per_col, columns=tech_col, values=acc_col).fillna(0)
        display(pivot.tail())
        pivot.plot(kind="area", stacked=True, figsize=(13,6))
        plt.title("Composición de accesos por tecnología")
        plt.xlabel("Período")
        plt.ylabel("Accesos")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()

        fiber_names = [x for x in pivot.columns if "fibra" in str(x).lower()]
        if fiber_names:
            fiber_share = pivot[fiber_names].sum(axis=1) / pivot.sum(axis=1).replace(0, np.nan) * 100
            display(fiber_share.rename("participacion_fibra_pct").to_frame())
            fiber_share.plot(figsize=(12,4), marker="o", title="Participación de fibra óptica")
            plt.ylabel("% de accesos tecnológicos")
            plt.xlabel("Período")
            plt.tight_layout()
            plt.show()
        else:
            print("No se encontró una categoría cuyo nombre contenga 'fibra'.")
    else:
        print("Faltan columnas de tecnología, accesos o período.")


## 11. Brechas provinciales de penetración

Se comparan las provincias en el último período disponible. Los indicadores de accesos cada 100 hogares/habitantes son tasas de penetración reportadas; **no equivalen necesariamente al porcentaje de hogares o personas únicas conectadas**.

In [ ]:
if "fact_penetracion_provincia" in data:
    df = data["fact_penetracion_provincia"].copy()
    prov_col = choose_col(df, ["provincia", "nombre_provincia"])
    per_col = resolve_period_column(df)
    metric_cols = [c for c in df.columns if "100_hogares" in c.lower() or "100_habitantes" in c.lower()
                   or "por_100_hogares" in c.lower() or "por_100_habitantes" in c.lower()]
    if prov_col and per_col and metric_cols:
        periods = df[per_col].dropna().astype(str).unique().tolist()
        last_period = sorted(periods)[-1] if periods else None
        latest = df[df[per_col].astype(str) == str(last_period)].copy()
        display(latest[[prov_col, per_col] + metric_cols].sort_values(metric_cols[0], ascending=False))
        for metric in metric_cols:
            latest.sort_values(metric).plot(kind="barh", x=prov_col, y=metric, legend=False,
                                            figsize=(9, max(4, len(latest)*.25)))
            plt.title(f"{metric} por provincia – {last_period}")
            plt.xlabel(metric)
            plt.ylabel("")
            plt.tight_layout()
            plt.show()
    else:
        print("No se identificaron las columnas necesarias para penetración provincial.")


## 12. Cruces entre variables

Ejemplos de cruces pertinentes: período × tecnología, provincia × penetración y provincia × fibra. Se muestran tablas/gráficos cuando las columnas existen.

In [ ]:
# Provincia x accesos (último período), para visualizar diferencias de escala
if "fact_accesos_provincia" in data:
    df = data["fact_accesos_provincia"].copy()
    prov = choose_col(df, ["provincia", "nombre_provincia"])
    per = resolve_period_column(df)
    acc = choose_col(df, ["accesos_total", "total_accesos", "accesos"])
    if prov and per and acc:
        last = sorted(df[per].dropna().astype(str).unique())[-1]
        cross = df[df[per].astype(str) == last].groupby(prov)[acc].sum().sort_values(ascending=False)
        display(cross.to_frame("accesos_total"))
        cross.sort_values().plot(kind="barh", figsize=(9, max(4, len(cross)*.25)), title=f"Accesos por provincia – {last}")
        plt.tight_layout()
        plt.show()

# Tecnología x período ya se analiza en la sección 10.


## 13. Correlaciones e hipótesis exploratoria

Hipótesis exploratoria: **las provincias con mayor participación de fibra óptica tienden a presentar mayor velocidad media**.

- H0: no existe asociación monotónica entre participación de fibra y velocidad media provincial.
- H1: existe asociación monotónica positiva.

Se utiliza Spearman como medida principal porque no exige relación lineal. Pearson se informa como complemento. La correlación no demuestra causalidad y debe calcularse sobre provincias con ambas variables disponibles.

In [ ]:
fiber_df = speed_df = None
if "fact_accesos_tecnologia" in data and "fact_velocidad_provincia" in data:
    ft = data["fact_accesos_tecnologia"].copy()
    fv = data["fact_velocidad_provincia"].copy()
    p_ft = choose_col(ft, ["provincia", "nombre_provincia"])
    t_ft = choose_col(ft, ["tecnologia", "nombre_tecnologia"])
    a_ft = choose_col(ft, ["accesos", "cantidad_accesos", "total"])
    p_fv = choose_col(fv, ["provincia", "nombre_provincia"])
    v_fv = choose_col(fv, ["mbps_media", "velocidad_media", "velocidad"])
    per_ft = resolve_period_column(ft)
    per_fv = resolve_period_column(fv)

    if all([p_ft, t_ft, a_ft, p_fv, v_fv, per_ft, per_fv]):
        last_ft = sorted(ft[per_ft].dropna().astype(str).unique())[-1]
        last_fv = sorted(fv[per_fv].dropna().astype(str).unique())[-1]
        ft_last = ft[ft[per_ft].astype(str) == last_ft].copy()
        ft_last["_es_fibra"] = ft_last[t_ft].astype(str).str.contains("fibra", case=False, na=False)
        agg = ft_last.groupby([p_ft, "_es_fibra"])[a_ft].sum().unstack(fill_value=0)
        if True not in agg.columns: agg[True] = 0
        if False not in agg.columns: agg[False] = 0
        agg["participacion_fibra_pct"] = agg[True] / (agg[True] + agg[False]).replace(0, np.nan) * 100
        speed_last = fv[fv[per_fv].astype(str) == last_fv].groupby(p_fv)[v_fv].mean()
        corr_df = agg[["participacion_fibra_pct"]].join(speed_last.rename("velocidad_media_mbps"), how="inner").dropna()
        display(corr_df)
        if len(corr_df) >= 3:
            display(corr_df.corr(method="spearman"))
            display(corr_df.corr(method="pearson"))
            corr_df.plot.scatter(x="participacion_fibra_pct", y="velocidad_media_mbps", figsize=(7,5))
            plt.title("Participación de fibra vs velocidad media provincial")
            plt.tight_layout()
            plt.show()
            print("Provincias incluidas:", len(corr_df))
            print("Spearman:", corr_df.corr(method="spearman").iloc[0,1])
            print("Pearson:", corr_df.corr(method="pearson").iloc[0,1])
        else:
            print("No hay suficientes provincias con ambas variables para correlacionar.")
    else:
        print("No se pudieron identificar todas las columnas para la hipótesis.")
else:
    print("Faltan tablas para evaluar la hipótesis.")


## 14. Detección de outliers mediante rango intercuartílico (IQR)

Se marcan observaciones fuera de `[Q1 - 1.5×IQR, Q3 + 1.5×IQR]`. Es un criterio estadístico exploratorio, no una prueba de error. En datos provinciales, diferencias reales de población o escala pueden producir valores extremos. No se eliminan registros.

In [ ]:
outlier_rows = []
for t, df in data.items():
    for c in df.select_dtypes(include="number").columns:
        if any(h in c.lower() for h in ID_HINTS):
            continue
        s = df[c].dropna()
        if len(s) < 4 or s.nunique() < 3:
            continue
        q1, q3 = s.quantile(.25), s.quantile(.75)
        iqr = q3 - q1
        if iqr == 0:
            lower, upper = q1, q3
        else:
            lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
        mask = df[c].notna() & ((df[c] < lower) | (df[c] > upper))
        outlier_rows.append({
            "tabla": t, "variable": c, "limite_inferior": lower,
            "limite_superior": upper, "outliers": int(mask.sum()),
            "porcentaje_outliers": round(mask.sum()/max(df[c].notna().sum(),1)*100,2)
        })
        if mask.any():
            print(f"\n{t}.{c}: {int(mask.sum())} posibles outliers")
            display(df.loc[mask].head(15))
outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary.sort_values("porcentaje_outliers", ascending=False) if not outlier_summary.empty else outlier_summary)


## 15. Análisis territorial de localidades (sin evolución temporal)

Las tablas de localidad del modelo no tienen período. Por eso se puede estudiar distribución/ranking territorial, pero no afirmar crecimiento o evolución histórica de cada localidad.

In [ ]:
for t in ["fact_accesos_tecnologia_localidad", "fact_accesos_categoria_localidad"]:
    if t not in data:
        continue
    df = data[t].copy()
    loc = choose_col(df, ["localidad", "nombre_localidad"])
    acc = choose_col(df, ["accesos", "cantidad_accesos", "total"])
    if loc and acc:
        ranking = df.groupby(loc)[acc].sum().sort_values(ascending=False).head(20)
        print(t, "– 20 localidades con más accesos")
        display(ranking.to_frame("accesos"))
        ranking.sort_values().plot(kind="barh", figsize=(9,6), title=f"Top 20 localidades – {t}")
        plt.xlabel("Accesos")
        plt.tight_layout()
        plt.show()


## 16. Identificación de patrones y primeras conclusiones exploratorias

Esta sección produce indicadores automáticos cuando las tablas y columnas están disponibles. Los resultados deben interpretarse junto con la cobertura de datos y las validaciones SQL; no son conclusiones causales.

In [ ]:
observaciones = []

# Patrón temporal de accesos
if "fact_accesos_provincia" in data:
    df = data["fact_accesos_provincia"]
    per = resolve_period_column(df)
    acc = choose_col(df, ["accesos_total", "total_accesos", "accesos"])
    if per and acc:
        ts = df.groupby(per)[acc].sum()
        if len(ts) >= 2 and ts.iloc[0] != 0:
            delta = (ts.iloc[-1] / ts.iloc[0] - 1) * 100
            observaciones.append(f"Los accesos totales variaron {delta:.1f}% entre el primer y último período disponible ({ts.index[0]} a {ts.index[-1]}).")
            observaciones.append(f"El máximo de accesos se registró en {ts.idxmax()} ({ts.max():,.0f}).")

# Desigualdad provincial de penetración
if "fact_penetracion_provincia" in data:
    df = data["fact_penetracion_provincia"]
    per = resolve_period_column(df)
    metric = choose_col(df, ["accesos_cada_100_hogares", "accesos_por_100_hogares"])
    prov = choose_col(df, ["provincia", "nombre_provincia"])
    if per and metric and prov:
        last = sorted(df[per].dropna().astype(str).unique())[-1]
        vals = df[df[per].astype(str) == last].groupby(prov)[metric].mean().dropna()
        if len(vals) >= 2:
            observaciones.append(f"En {last}, la brecha entre la provincia con mayor y menor valor de {metric} fue {vals.max()-vals.min():.2f} puntos de la tasa reportada.")
            observaciones.append(f"Mayor valor: {vals.idxmax()} ({vals.max():.2f}); menor valor: {vals.idxmin()} ({vals.min():.2f}).")

# Concentración territorial del ranking de localidades
for t in ["fact_accesos_tecnologia_localidad", "fact_accesos_categoria_localidad"]:
    if t in data:
        df = data[t]
        loc = choose_col(df, ["localidad", "nombre_localidad"])
        acc = choose_col(df, ["accesos", "cantidad_accesos", "total"])
        if loc and acc:
            s = df.groupby(loc)[acc].sum().sort_values(ascending=False)
            total = s.sum()
            if total > 0:
                top10 = s.head(10).sum()/total*100
                observaciones.append(f"En {t}, las 10 localidades con más accesos concentran {top10:.1f}% del total registrado en esa tabla.")

print("PRIMERAS OBSERVACIONES EXPLORATORIAS")
if observaciones:
    for i, obs in enumerate(observaciones, 1):
        print(f"{i}. {obs}")
else:
    print("No se pudieron generar observaciones automáticas: revisá el mapeo de columnas.")

print("\nCompletar luego con interpretación del equipo:")
print("- ¿Qué patrones son consistentes con los objetivos del trabajo?")
print("- ¿Qué provincias o tecnologías muestran comportamientos destacados?")
print("- ¿Qué resultados requieren validar cobertura o calidad?")
print("- ¿Qué hipótesis quedan respaldadas, rechazadas o pendientes?")


## 17. Exportación de resultados

Se exportan estadísticas descriptivas y resumen de outliers a CSV para incorporarlos a la documentación del trabajo.

In [ ]:
OUTPUT_DIR = Path("salidas_eda")
OUTPUT_DIR.mkdir(exist_ok=True)

if not numeric_stats.empty:
    numeric_stats.to_csv(OUTPUT_DIR / "estadisticas_descriptivas.csv", index=False, encoding="utf-8-sig")
if not outlier_summary.empty:
    outlier_summary.to_csv(OUTPUT_DIR / "resumen_outliers_iqr.csv", index=False, encoding="utf-8-sig")

with open(OUTPUT_DIR / "primeras_observaciones.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(f"- {x}" for x in observaciones))

print("Resultados guardados en:", OUTPUT_DIR.resolve())


## 18. Limitaciones metodológicas para dejar explícitas en el informe

1. La velocidad nacional ponderada es una estimación propia si se calcula ponderando velocidades provinciales por accesos; no debe presentarse como un indicador nacional publicado directamente por ENACOM.
2. Las categorías de velocidad se mantienen como etiquetas originales; no se asignan rangos/unidades que la fuente no documente.
3. Los datos de localidades no tienen período, por lo que permiten comparaciones transversales, no series temporales.
4. Accesos cada 100 hogares/habitantes no equivale necesariamente a hogares/personas únicas conectadas.
5. La correlación entre fibra y velocidad no demuestra causalidad.
6. Los outliers IQR son observaciones para revisar, no registros que deban eliminarse automáticamente.
7. En comparaciones entre tablas hay que revisar cobertura de provincias y períodos; los JOIN pueden excluir registros sin correspondencia.
8. Las cinco tecnologías provinciales modeladas en `fact_accesos_tecnologia` son ADSL, Cablemodem, Fibra Óptica, Wireless y Otros. No se deben interpretar tecnologías ausentes como si tuvieran valor cero histórico.


## 19. Cierre

In [ ]:
conn.close()
print("Análisis finalizado. Revisá los avisos de columnas no identificadas y ajustá MAP si fuera necesario.")
